In [37]:

import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import dgl
import torch
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# Import graph-sc modules
import train
import models

# Set random seeds
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


# Load expression data (genes x cells) and transpose
df = pd.read_csv("../data/sce_sc_10x_qc_counts_matrix.csv", index_col=0)
X = df.T.values.astype(np.float32)  # Convert to float to avoid casting errors

# Load ground truth
groundtruth = pd.read_csv('../data/groundtruth.csv')
Y = groundtruth['x'].values

le = LabelEncoder()
Y_encoded = le.fit_transform(Y)

print(f"Data shape: {X.shape[0]} cells x {X.shape[1]} genes")
print(f"Cell types: {le.classes_}")
print(f"Sparsity: {np.where(X == 0)[0].shape[0] / (X.shape[0] * X.shape[1]):.4f}")

# ============================================================================
# 2. Set parameters 
# ============================================================================

normalize_weights = "log_per_cell"

n_layers = 1
hidden_dim = 200
hidden = [300]
nb_genes = 500
activation = F.relu
epochs = 10
batch_size = 128 
pca_size = 50

'''

nb_genes = 500  # Number of highly variable genes
pca_size = 50    # PCA dimensions
normalize_weights = "log_per_cell"
n_layers = 2
hidden_dim = 128
hidden = [128]
activation = F.relu
epochs = 10
batch_size = 128

'''

n_clusters = len(np.unique(Y_encoded))

print(f"\nParameters:")
print(f"  HVGs: {nb_genes}")
print(f"  PCA dims: {pca_size}")
print(f"  Epochs: {epochs}")
print(f"  Clusters: {n_clusters}")

# ============================================================================
# 3. Preprocess: filter genes
# ============================================================================
print("\n" + "="*60)
print("Preprocessing...")
print("="*60)

genes_idx, cells_idx = train.filter_data(X, highly_genes=nb_genes)
X = X[cells_idx][:, genes_idx]
Y_encoded = Y_encoded[cells_idx]

print(f"After filtering: {X.shape[0]} cells x {X.shape[1]} genes")

# ============================================================================
# 4. Create graph
# ============================================================================
print("\n" + "="*60)
print("Creating graph...")
print("="*60)

graph = train.make_graph(
    X,
    Y_encoded,  # Ground truth for validation
    dense_dim=pca_size,
    normalize_weights=normalize_weights,
)

labels = graph.ndata["label"]
train_ids = np.where(labels != -1)[0]

print(f"Graph created with {graph.num_nodes()} nodes and {graph.num_edges()} edges")

# ============================================================================
# 5. Create data loader
# ============================================================================
sampler = dgl.dataloading.MultiLayerFullNeighborSampler(n_layers)

dataloader = dgl.dataloading.DataLoader(
    graph,
    train_ids,
    sampler,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=10,
)

# ============================================================================
# 6. Create model
# ============================================================================
print("\n" + "="*60)
print("Creating model...")
print("="*60)

device = train.get_device(use_cpu=not torch.cuda.is_available())
print(f"Using device: {device}")

model = models.GCNAE(
    in_feats=pca_size,
    n_hidden=hidden_dim,
    n_layers=n_layers,
    activation=activation,
    dropout=0.1,
    hidden=hidden,
).to(device)

print(f"Model: {model}")

# ============================================================================
# 7. Train model
# ============================================================================
print("\n" + "="*60)
print("Training Graph-sc...")
print("="*60)

optim = torch.optim.Adam(model.parameters(), lr=1e-3)

results = train.train(
    model,
    optim,
    epochs,
    dataloader,
    n_clusters,
    plot=False,
    save=False,
    cluster=["KMeans"],  # Can also add "Leiden"
    use_cpu=not torch.cuda.is_available()
)
 
# ============================================================================
# 8. Display and save results
# ============================================================================
print("\n" + "="*60)
print("Graph-sc Results:")
print("="*60)
print(f"  ARI: {results['kmeans_ari']:.4f}")
print(f"  NMI: {results['kmeans_nmi']:.4f}")
print(f"  Silhouette: {results['kmeans_sil']:.4f}")
print(f"  Calinski-Harabasz: {results['kmeans_cal']:.4f}")


Loading data...
Data shape: 902 cells x 16468 genes
Cell types: ['H1975' 'H2228' 'HCC827']
Sparsity: 0.4502

Parameters:
  HVGs: 500
  PCA dims: 50
  Epochs: 10
  Clusters: 3

Preprocessing...
After filtering: 902 cells x 500 genes

Creating graph...
Graph created with 1402 nodes and 176522 edges

Creating model...
Using device: cpu
Model: GCNAE(
  (dropout): Dropout(p=0.1, inplace=False)
  (layer1): WeightedGraphConv(in=50, out=200, normalization=both, activation=<function relu at 0x14775f64ee50>)
  (decoder): InnerProductDecoder()
  (encoder): Sequential(
    (0): Linear(in_features=200, out_features=300, bias=True)
  )
)

Training Graph-sc...


100%|██████████| 10/10 [00:08<00:00,  1.15it/s]


ARI 0.9797, 0.38073548674583435

Graph-sc Results:
  ARI: 0.9797
  NMI: 0.9626
  Silhouette: 0.3807
  Calinski-Harabasz: 453.5229


In [27]:

import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import dgl
import torch
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# Import graph-sc modules
import train
import models

# Set random seeds
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)


# Load expression data (genes x cells) and transpose
df = pd.read_csv("../data/sce_sc_10x_5cl_qc_counts_matrix.csv", index_col=0)
X = df.T.values.astype(np.float32)  # Convert to float to avoid casting errors

# Load ground truth
groundtruth = pd.read_csv('../data/groundtruth_5cl.csv')
Y = groundtruth['x'].values

le = LabelEncoder()
Y_encoded = le.fit_transform(Y)

print(f"Data shape: {X.shape[0]} cells x {X.shape[1]} genes")
print(f"Cell types: {le.classes_}")
print(f"Sparsity: {np.where(X == 0)[0].shape[0] / (X.shape[0] * X.shape[1]):.4f}")

# ============================================================================
# 2. Set parameters 
# ============================================================================
nb_genes = 500  # Number of highly variable genes
pca_size = 50    # PCA dimensions
normalize_weights = "log_per_cell"
n_layers = 2
hidden_dim = 128
hidden = [128]
activation = F.relu
epochs = 100
batch_size = 128
n_clusters = len(np.unique(Y_encoded))

print(f"\nParameters:")
print(f"  HVGs: {nb_genes}")
print(f"  PCA dims: {pca_size}")
print(f"  Epochs: {epochs}")
print(f"  Clusters: {n_clusters}")

# ============================================================================
# 3. Preprocess: filter genes
# ============================================================================

genes_idx, cells_idx = train.filter_data(X, highly_genes=nb_genes)
X = X[cells_idx][:, genes_idx]
Y_encoded = Y_encoded[cells_idx]

print(f"After filtering: {X.shape[0]} cells x {X.shape[1]} genes")

# ============================================================================
# 4. Create graph
# ============================================================================
print("\n" + "="*60)
print("Creating graph...")
print("="*60)

graph = train.make_graph(
    X,
    Y_encoded,  # Ground truth for validation
    dense_dim=pca_size,
    normalize_weights=normalize_weights,
)

labels = graph.ndata["label"]
train_ids = np.where(labels != -1)[0]

print(f"Graph created with {graph.num_nodes()} nodes and {graph.num_edges()} edges")

# ============================================================================
# 5. Create data loader
# ============================================================================
sampler = dgl.dataloading.MultiLayerFullNeighborSampler(n_layers)

dataloader = dgl.dataloading.DataLoader(
    graph,
    train_ids,
    sampler,
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
    num_workers=0,
)

# ============================================================================
# 6. Create model
# ============================================================================

device = train.get_device(use_cpu=not torch.cuda.is_available())
print(f"Using device: {device}")

model = models.GCNAE(
    in_feats=pca_size,
    n_hidden=hidden_dim,
    n_layers=n_layers,
    activation=activation,
    dropout=0.1,
    hidden=hidden,
).to(device)

print(f"Model: {model}")

# ============================================================================
# 7. Train model
# ============================================================================


optim = torch.optim.Adam(model.parameters(), lr=1e-5)

results = train.train(
    model,
    optim,
    epochs,
    dataloader,
    n_clusters,
    plot=False,
    save=False,
    cluster=["KMeans"],  # Can also add "Leiden"
    use_cpu=not torch.cuda.is_available()
)

# ============================================================================
# 8. Display and save results
# ============================================================================
print("\n" + "="*60)
print("Graph-sc Results:")
print("="*60)
print(f"  ARI: {results['kmeans_ari']:.4f}")
print(f"  NMI: {results['kmeans_nmi']:.4f}")
print(f"  Silhouette: {results['kmeans_sil']:.4f}")
print(f"  Calinski-Harabasz: {results['kmeans_cal']:.4f}")


Loading data...
Data shape: 3918 cells x 11786 genes
Cell types: ['A549' 'H1975' 'H2228' 'H838' 'HCC827']
Sparsity: 0.6301

Parameters:
  HVGs: 500
  PCA dims: 50
  Epochs: 100
  Clusters: 5

Preprocessing...
After filtering: 3918 cells x 500 genes

Creating graph...
Graph created with 4418 nodes and 507634 edges

Creating model...
Using device: cpu
Model: GCNAE(
  (dropout): Dropout(p=0.1, inplace=False)
  (layer1): WeightedGraphConv(in=50, out=128, normalization=both, activation=<function relu at 0x14775f64ee50>)
  (layer2): WeightedGraphConv(in=128, out=128, normalization=both, activation=<function relu at 0x14775f64ee50>)
  (decoder): InnerProductDecoder()
  (encoder): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
  )
)

Training Graph-sc...


100%|██████████| 100/100 [00:46<00:00,  2.16it/s]


ARI 0.9872, 0.5782894492149353

Graph-sc Results:
  ARI: 0.9872
  NMI: 0.9766
  Silhouette: 0.5783
  Calinski-Harabasz: 3095.1103


In [28]:

# Save results
import os
os.makedirs('../results', exist_ok=True)

results_df = pd.DataFrame({
    'Method': ['Graph-sc'],
    'ARI': [results['kmeans_ari']],
    'NMI': [results['kmeans_nmi']],
    'Silhouette': [results['kmeans_sil']],
    'Calinski': [results['kmeans_cal']],
    'N_Clusters': [n_clusters]
})

#results_df.to_csv('../results/graphsc_scMixology_results.csv', index=False)
results_df.to_csv('../results/graphsc_scMixology_5cl_results.csv', index=False)
print(results_df)

     Method     ARI     NMI  Silhouette     Calinski  N_Clusters
0  Graph-sc  0.9872  0.9766    0.578289  3095.110277           5


In [1]:
import nbformat

notebook_filename = "scMixology_Graphsc.ipynb"

with open(notebook_filename, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

for cell in nb.cells:
    print(f"--- {cell.cell_type.upper()} CELL ---")
    print(cell.source)
    print("\n")

--- CODE CELL ---
"""
Run Graph-sc clustering using their actual implementation
Based on their Main.ipynb notebook
Run this from the graph-sc directory
"""

import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import dgl
import torch
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

# Import graph-sc modules
import train
import models

# Set random seeds
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# ============================================================================
# 1. Load your data
# ============================================================================
print("="*60)
print("Loading data...")
print("="*60)

# Load expression data (genes x cells) and transpose
df = pd.read_csv("../data/sce_sc_10x_qc_counts_matrix.csv", index_col=0)
X = df.T.values.astype(np.float32)  # Convert to floa

In [1]:
"""
Run Graph-sc clustering with multiple runs and datasets
Returns 5 metrics: ARI, NMI, AMI, ACC, F1
"""

import sys
sys.path.append("..")
import os
import numpy as np
import pandas as pd
import dgl
import torch
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.metrics import adjusted_mutual_info_score, f1_score, accuracy_score
from scipy.optimize import linear_sum_assignment
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# Import graph-sc modules
import train
import models


def cluster_accuracy(y_true, y_pred):

    y_true = np.array(y_true).astype(np.int64)
    y_pred = np.array(y_pred).astype(np.int64)
    
    assert y_pred.size == y_true.size
    
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    
    return w[row_ind, col_ind].sum() / y_pred.size


def compute_metrics(y_true, y_pred):
    acc = cluster_accuracy(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    ari = adjusted_rand_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    
    y_true = np.array(y_true).astype(np.int64)
    y_pred = np.array(y_pred).astype(np.int64)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    
    # Create mapping from pred to true
    mapping = {row_ind[i]: col_ind[i] for i in range(len(row_ind))}
    y_pred_mapped = np.array([mapping.get(p, p) for p in y_pred])
    f1 = f1_score(y_true, y_pred_mapped, average='weighted')
    
    return acc, nmi, ari, ami, f1


def load_3cl_data():
    """Load the 3-cluster dataset"""
    df = pd.read_csv("../data/sce_sc_10x_qc_counts_matrix.csv", index_col=0)
    X = df.T.values.astype(np.float32)
    groundtruth = pd.read_csv('../data/groundtruth.csv')
    Y = groundtruth['x'].values
    return X, Y, "3cl"


def load_5cl_data():
    """Load the 5-cluster dataset"""
    df = pd.read_csv("../data/sce_sc_10x_5cl_qc_counts_matrix.csv", index_col=0)
    X = df.T.values.astype(np.float32)
    groundtruth = pd.read_csv('../data/groundtruth_5cl.csv')
    Y = groundtruth['x'].values
    return X, Y, "5cl"


def run_single_experiment(X, Y_encoded, n_clusters, run_idx, dataset_name,
                          nb_genes=500, pca_size=50, epochs=100, batch_size=128,
                          n_layers=1, hidden_dim=200, hidden=[300], lr=1e-3):
    """Run a single Graph-sc experiment"""
    
    # Set different random seed for each run
    seed = 42 + run_idx
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    X_run = X.copy()
    Y_run = Y_encoded.copy()
    
    # Filter genes
    genes_idx, cells_idx = train.filter_data(X_run, highly_genes=nb_genes)
    X_run = X_run[cells_idx][:, genes_idx]
    Y_run = Y_run[cells_idx]
    
    # Create graph
    graph = train.make_graph(
        X_run,
        Y_run,
        dense_dim=pca_size,
        normalize_weights="log_per_cell",
    )
    
    labels = graph.ndata["label"]
    train_ids = np.where(labels != -1)[0]
    
    # Create data loader
    sampler = dgl.dataloading.MultiLayerFullNeighborSampler(n_layers)
    dataloader = dgl.dataloading.DataLoader(
        graph,
        train_ids,
        sampler,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0,
    )
    
    # Create model
    device = train.get_device(use_cpu=not torch.cuda.is_available())
    model = models.GCNAE(
        in_feats=pca_size,
        n_hidden=hidden_dim,
        n_layers=n_layers,
        activation=F.relu,
        dropout=0.1,
        hidden=hidden,
    ).to(device)
    
    # Train
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    results = train.train(
        model,
        optim,
        epochs,
        dataloader,
        n_clusters,
        plot=False,
        save=False,
        cluster=["KMeans"],
        use_cpu=not torch.cuda.is_available()
    )
    
    # Get predicted labels from results (kmeans_pred is returned by train.train)
    y_pred = results['kmeans_pred']
    
    # Compute all 5 metrics
    acc, nmi, ari, ami, f1 = compute_metrics(Y_run, y_pred)
    
    print(f"  Run {run_idx + 1} ({dataset_name}): ACC={acc:.4f}, NMI={nmi:.4f}, ARI={ari:.4f}, AMI={ami:.4f}, F1={f1:.4f}")
    
    return acc, nmi, ari, ami, f1


def run_multiple_experiments(X, Y, dataset_name, n_runs=5, n_jobs=-1, **kwargs):
    """Run multiple experiments in parallel"""
    
    # Encode labels
    le = LabelEncoder()
    Y_encoded = le.fit_transform(Y)
    n_clusters = len(np.unique(Y_encoded))
    
    print(f"\n{'='*60}")
    print(f"Running {n_runs} experiments for {dataset_name} (parallel, n_jobs={n_jobs})")
    print(f"Data shape: {X.shape[0]} cells x {X.shape[1]} genes")
    print(f"Clusters: {n_clusters}")
    print(f"{'='*60}")
    
    # Run experiments in parallel
    results_list = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_single_experiment)(
            X, Y_encoded, n_clusters, run_idx, dataset_name, **kwargs
        )
        for run_idx in range(n_runs)
    )
    
    # Collect results
    results = {
        'run': [],
        'ACC': [],
        'NMI': [],
        'ARI': [],
        'AMI': [],
        'F1': []
    }
    
    for run_idx, (acc, nmi, ari, ami, f1) in enumerate(results_list):
        results['run'].append(run_idx + 1)
        results['ACC'].append(acc)
        results['NMI'].append(nmi)
        results['ARI'].append(ari)
        results['AMI'].append(ami)
        results['F1'].append(f1)
    
    return pd.DataFrame(results)


def compute_summary(results_df):
    """Compute summary statistics"""
    metrics = ['ACC', 'NMI', 'ARI', 'AMI', 'F1']
    summary = {'Metric': [], 'Mean': [], 'Std': [], 'Min': [], 'Max': []}
    
    for metric in metrics:
        summary['Metric'].append(metric)
        summary['Mean'].append(results_df[metric].mean())
        summary['Std'].append(results_df[metric].std())
        summary['Min'].append(results_df[metric].min())
        summary['Max'].append(results_df[metric].max())
    
    return pd.DataFrame(summary)


def main():
    # Configuration
    N_RUNS = 5
    N_JOBS = -1  # -1 = all cores
    EPOCHS = 100
    
    # Create results directory
    os.makedirs('../results', exist_ok=True)
    
    # Dataset configurations
    datasets = [
        ('3cl', load_3cl_data, {'nb_genes': 500, 'pca_size': 50, 'epochs': EPOCHS,
                                'n_layers': 1, 'hidden_dim': 200, 'hidden': [300], 'lr': 1e-3}),
        ('5cl', load_5cl_data, {'nb_genes': 500, 'pca_size': 50, 'epochs': EPOCHS,
                                'n_layers': 2, 'hidden_dim': 128, 'hidden': [128], 'lr': 1e-5}),
    ]
    
    all_summaries = {}
    
    for dataset_name, load_func, params in datasets:

        try:
            X, Y, name = load_func()
            
            # Run experiments
            results_df = run_multiple_experiments(
                X, Y, dataset_name,
                n_runs=N_RUNS,
                n_jobs=N_JOBS,
                **params
            )
            
            # Compute summary
            summary_df = compute_summary(results_df)
            all_summaries[dataset_name] = summary_df
            
            # Save individual runs
            results_path = f'../results/graphsc_{dataset_name}_all_runs.csv'
            results_df.to_csv(results_path, index=False)
            print(f"\nIndividual runs saved to: {results_path}")
            
            # Save summary
            summary_path = f'../results/graphsc_{dataset_name}_summary.csv'
            summary_df.to_csv(summary_path, index=False)
            print(f"Summary saved to: {summary_path}")
            
            # Print summary
            print(f"\n{'='*60}")
            print(f"Summary for {dataset_name}:")
            print(f"{'='*60}")
            print(summary_df.to_string(index=False))
            
        except FileNotFoundError as e:
            print(f"Error loading {dataset_name} data: {e}")
            continue
    
    # Print combined summary
    print(f"\n{'#'*60}")
    print("# FINAL SUMMARY - ALL DATASETS")
    print(f"{'#'*60}")
    for dataset_name, summary_df in all_summaries.items():
        print(f"\n{dataset_name}:")
        print(summary_df.to_string(index=False))
    
    # Save combined summary
    if all_summaries:
        combined = []
        for dataset_name, summary_df in all_summaries.items():
            summary_df = summary_df.copy()
            summary_df.insert(0, 'Dataset', dataset_name)
            combined.append(summary_df)
        combined_df = pd.concat(combined, ignore_index=True)
        combined_path = '../results/graphsc_combined_summary.csv'
        combined_df.to_csv(combined_path, index=False)
        print(f"\nCombined summary saved to: {combined_path}")


if __name__ == "__main__":
    main()


############################################################
# Processing 3cl dataset
############################################################

Running 5 experiments for 3cl (parallel, n_jobs=-1)
Data shape: 902 cells x 16468 genes
Clusters: 3


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._reg

ARI 0.9599, 0.13516873121261597
  Run 3 (3cl): ACC=0.9867, NMI=0.9337, ARI=0.9599, AMI=0.9336, F1=0.9867
ARI 0.6399, 0.1770835965871811
  Run 4 (3cl): ACC=0.8326, NMI=0.7250, ARI=0.6399, AMI=0.7244, F1=0.8246


/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   40.8s remaining:  1.0min
100%|██████████| 100/100 [00:28<00:00,  3.53it/s]
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   40.9s remaining:   27.3s


ARI 0.9866, 0.16775092482566833
  Run 1 (3cl): ACC=0.9956, NMI=0.9756, ARI=0.9866, AMI=0.9756, F1=0.9956


...

[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:  1.9min remaining:  1.3min
100%|██████████| 100/100 [01:45<00:00,  1.06s/it]
/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
100%|██████████| 100/100 [01:46<00:00,  1.06s/it]


ARI 0.9883, 0.5729776620864868
  Run 2 (5cl): ACC=0.9949, NMI=0.9782, ARI=0.9883, AMI=0.9782, F1=0.9949


/home/he.wan1/.conda/envs/py38/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


ARI 0.9892, 0.5705904364585876
  Run 3 (5cl): ACC=0.9952, NMI=0.9782, ARI=0.9892, AMI=0.9782, F1=0.9952

Individual runs saved to: ../results/graphsc_5cl_all_runs.csv
Summary saved to: ../results/graphsc_5cl_summary.csv

Summary for 5cl:
Metric     Mean      Std      Min      Max
   ACC 0.994997 0.000588 0.994385 0.995916
   NMI 0.978243 0.001930 0.976625 0.981417
   ARI 0.988439 0.001466 0.987020 0.990562
   AMI 0.978214 0.001933 0.976594 0.981392
    F1 0.994999 0.000587 0.994386 0.995917

############################################################
# FINAL SUMMARY - ALL DATASETS
############################################################

3cl:
Metric     Mean      Std      Min      Max
   ACC 0.861197 0.127483 0.702882 0.995565
   NMI 0.765364 0.188253 0.524238 0.975643
   ARI 0.702033 0.265008 0.377498 0.986579
   AMI 0.764852 0.188679 0.523154 0.975594
    F1 0.851260 0.138005 0.678462 0.995572

5cl:
Metric     Mean      Std      Min      Max
   ACC 0.994997 0.000588 0.994385 0.9

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.0min remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.0min finished
